# Generate synthetic player-360 churn data

This notebook builds a small synthetic dataset that mimics player-360 features (engagement, tech crashes, payment cadence) with churn as the target. It's used as the input for the WIDS example configs in `examples/wids/`.

## Setup

We use a small, focused dependency set:

- `pandas` / `numpy` for tabular handling and RNG seeding.
- `pgmpy.models.BayesianNetwork` to declare the DAG over player features.
- `pgmpy.factors.discrete.TabularCPD` to attach a conditional probability   table to each node.
- `pgmpy.sampling.BayesianModelSampling` for forward (ancestral) sampling   from the fully specified network.

All dependencies are managed by Poetry; there are no `!pip install` calls in this notebook. Parquet I/O goes through `pyarrow`, which is already a project dependency.

In [ ]:
import numpy as np
import pandas as pd
from pgmpy.factors.discrete import TabularCPD
from pgmpy.models import BayesianNetwork
from pgmpy.sampling import BayesianModelSampling

## Define the causal structure

We model a small DAG over player-360 features. Each edge `parent -> child` encodes that the parent has direct causal influence on the child *conditional on the rest of the parents in the network*.

**Engagement layer.** Three exogenous engagement signals feed into a single composite `game_activity` node:

- `daily_login_rewards` -> `game_activity`
- `level_completion_bonuses` -> `game_activity`
- `special_event_notifications` -> `game_activity`

**Churn layer.** Churn is driven by activity, tech quality, and payment cadence:

- `game_activity` -> `churn` — disengaged players are likelier to leave.
- `tech_crashes` -> `churn` — frequent crashes degrade the experience.
- `days_between_login` -> `churn` — long gaps signal waning interest.
- `days_between_payment` -> `churn` — payment-cadence drift is a known   downstream signal of churn intent.

There is exactly one structure definition below — earlier drafts of this notebook iterated through several near-duplicate variants which have been collapsed.

In [ ]:
edges = [
    ('daily_login_rewards', 'game_activity'),
    ('level_completion_bonuses', 'game_activity'),
    ('special_event_notifications', 'game_activity'),
    ('game_activity', 'churn'),
    ('tech_crashes', 'churn'),
    ('days_between_login', 'churn'),
    ('days_between_payment', 'churn'),
]

model = BayesianNetwork(edges)
model.nodes()

## Specify the conditional probability distributions (CPDs)

A **CPD** is a table that encodes `P(child | parents)`: for every combination of parent states, it lists the probability of each child state. For a binary child with `k` binary parents, that's a `2 x 2**k` table whose columns must each sum to 1.

All variables here are binarised to keep the network compact and the tutorial readable:

- Engagement signals (`daily_login_rewards`, `level_completion_bonuses`,   `special_event_notifications`): `0` = low / absent, `1` = high / present.
- `game_activity`: `0` = low engagement, `1` = high engagement. Three   binary parents -> `evidence_card=[2, 2, 2]` -> 8 columns.
- `tech_crashes`, `days_between_login`, `days_between_payment`:   `0` = low (good), `1` = high (bad). Marginals are tilted toward the   benign state because most player-days are normal.
- `churn`: `0` = retained, `1` = churned. Four binary parents ->   `evidence_card=[2, 2, 2, 2]` -> 16 columns.

The churn CPD below is the final, definitive version. The original notebook contained three or four near-duplicate iterations ("adjusting the CPD for churn based on correct evidence cards…") that were all drafts of this same table; they have been removed.

In [ ]:
# Exogenous engagement priors.
cpd_daily_login_rewards = TabularCPD(
    variable='daily_login_rewards', variable_card=2,
    values=[[0.5], [0.5]],
)
cpd_level_completion_bonuses = TabularCPD(
    variable='level_completion_bonuses', variable_card=2,
    values=[[0.5], [0.5]],
)
cpd_special_event_notifications = TabularCPD(
    variable='special_event_notifications', variable_card=2,
    values=[[0.7], [0.3]],
)

# Composite engagement node: P(game_activity | three engagement parents).
# Columns enumerate parent states in the order
# (daily_login_rewards, level_completion_bonuses, special_event_notifications).
cpd_game_activity = TabularCPD(
    variable='game_activity', variable_card=2,
    values=[
        [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],  # P(low activity | parents)
        [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2],  # P(high activity | parents)
    ],
    evidence=['daily_login_rewards',
              'level_completion_bonuses',
              'special_event_notifications'],
    evidence_card=[2, 2, 2],
)

# Tech and cadence priors (most player-days are healthy).
cpd_tech_crashes = TabularCPD(
    variable='tech_crashes', variable_card=2,
    values=[[0.8], [0.2]],
)
cpd_days_between_login = TabularCPD(
    variable='days_between_login', variable_card=2,
    values=[[0.6], [0.4]],
)
cpd_days_between_payment = TabularCPD(
    variable='days_between_payment', variable_card=2,
    values=[[0.7], [0.3]],
)

# Churn CPD: P(churn | game_activity, tech_crashes,
#                    days_between_login, days_between_payment).
# 4 binary parents => 2**4 = 16 columns. Each column sums to 1.
cpd_churn = TabularCPD(
    variable='churn', variable_card=2,
    values=[
        [0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15,
         0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75],  # P(retained | ...)
        [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85,
         0.95, 0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25],  # P(churned  | ...)
    ],
    evidence=['game_activity', 'tech_crashes',
              'days_between_login', 'days_between_payment'],
    evidence_card=[2, 2, 2, 2],
)

model.add_cpds(
    cpd_daily_login_rewards,
    cpd_level_completion_bonuses,
    cpd_special_event_notifications,
    cpd_game_activity,
    cpd_tech_crashes,
    cpd_days_between_login,
    cpd_days_between_payment,
    cpd_churn,
)

assert model.check_model()

## Sample from the model

With the network fully specified we can draw IID samples by forward (ancestral) sampling: pgmpy walks the DAG in topological order, sampling each node from its CPD given already-sampled parents.

We pin `seed_random_state=42` for reproducibility, then attach a synthetic `user_id` and a daily `timestamp` so the dataset looks like a player-360 snapshot.

In [ ]:
N = 1000

sampler = BayesianModelSampling(model)
df = sampler.forward_sample(size=N, seed=42)

df['user_id'] = np.arange(1, N + 1)
df['timestamp'] = pd.date_range('2023-01-01', periods=N, freq='D')

df.head()

## Save to parquet

We write the dataset to `data/synthetic_churn.parquet` at the project root. The path below uses `..` because this notebook lives in `notebooks/`. `pyarrow` (a project dependency) handles the parquet encoding — no extra installs needed.

In [ ]:
from pathlib import Path

out = Path("../data/synthetic_churn.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out, index=False)
print(f"Wrote {len(df)} rows -> {out.resolve()}")

## Next steps

Run `poetry run infer-bn --config_file examples/wids/config_simulation.yaml` from the repo root to fit a Bayesian network on this synthetic dataset.